# Units

MobsPy uses the Pint library for unit-aware modeling. You can specify concentrations, volumes, and rate constants with physical units. MobsPy validates dimensional consistency at compile time.

When you provide units on `duration` or `volume`, MobsPy infers the model's time and volume units and preserves your rate constants in those units. For example, a rate of `0.5 / u.hour` with `duration = 10 * u.hour` stays as `0.5` in the compiled SBML, not converted to per-second.

In [ ]:
from mobspy import *

## The Unit Object

Access units through `u`. Assign concentrations as initial conditions instead of raw counts. The expression `10 / u.liter` means 10 molecules per liter.

In [ ]:
A = BaseSpecies()
A(10 / u.liter)

sim = Simulation(A)
print(sim.compile())

## Volume and Concentration

The `volume` parameter sets the SBML compartment size. Species symbols in rate laws represent concentrations (amount / volume). With 10 per liter and a volume of 2 liters, the model starts with 20 molecules, and the species concentration is 10/L.

In [ ]:
A2 = BaseSpecies()
A2(10 / u.liter)

A2 >> Zero @ (0.1 / u.second)

sim2 = Simulation(A2)
sim2.volume = 2 * u.liter
print(sim2.compile())

## Rate Constants with Units

For a first-order reaction, the rate constant has units of $1/[\text{Time}]$. MobsPy checks that the dimensions match the reaction order.

In [ ]:
B = BaseSpecies()
B(50 / u.liter)

# First-order decay: rate has units 1/time
B >> Zero @ (0.1 / u.second)

sim3 = Simulation(B)
sim3.volume = 1 * u.liter
print(sim3.compile())

## Second-Order Rates

For a second-order reaction (`A + B >> C`), the rate constant must include volume dimensions: $1 / ([\text{Time}] \cdot [\text{Volume}])$. This ensures dimensional consistency when multiplied by two concentrations.

In [ ]:
C, D, E = BaseSpecies()
C(10 / u.liter)
D(20 / u.liter)

C + D >> E @ (1e-3 * u.liter / u.second)

sim4 = Simulation(C | D | E)
sim4.volume = 1 * u.liter
print(sim4.compile())

## Dimensional Consistency Checks

MobsPy detects dimensional mismatches at compile time. For example, assigning a 3D volume to a model whose concentrations are defined per area (2D) raises an error.

In [ ]:
F, G = BaseSpecies()
F(10 / u.meter**2)
G(5 / u.meter**2)

F + G >> F @ 1

try:
    sim5 = Simulation(F | G)
    sim5.volume = 3 * u.meter**3  # 3D volume for a 2D model
    sim5.compile()
except Exception as e:
    print(f"Caught error: {type(e).__name__}")

## Output Units

MobsPy infers the model's time unit from `duration` (or from rate constants if `duration` has no units). The output time axis is in the model's time unit by default. You can override with `unit_x` (time axis) and `unit_y` (species axis). The `output_concentration` flag controls whether results are in concentration or raw counts.

In [ ]:
H = BaseSpecies()
H(100 * u.mol)

H >> Zero @ (1 / u.hour)

sim6 = Simulation(H)
sim6.duration = 10 * u.hour
sim6.unit_x = 1 * u.hour
sim6.unit_y = 1 * u.mol
sim6.output_concentration = False
sim6.plot_data = False
sim6.run()

print("Time (first 5):", sim6.fres["Time"][:5])
print("H (first 5):", sim6.fres[H][:5])